In [1]:
import pandas as pd
import numpy as np

# Feature Engineering & Encoding
Generating quantitative signals (RSI) and qualitative candlestick patterns (Doji, Engulfing, Hammer), followed by strict garbage-collection and one-hot encoding for ML ingestion.

In [2]:
df = pd.read_csv('../data/processed/base_market_data.csv')

In [3]:
df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI


In [4]:
feature_df = df.copy()

In [5]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI


In [6]:
feature_df['Next_Close'] = feature_df.groupby('Ticker_Name')['Close'].shift(-1)

In [7]:
feature_df['Target'] = (feature_df['Next_Close'] > feature_df['Close']).astype(int)

In [8]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1


In [9]:
feature_df['Price_Change'] = feature_df.groupby('Ticker_Name')['Close'].diff()

In [10]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195


In [11]:
feature_df['Gains'] = np.where(feature_df['Price_Change'] < 0 , 0, feature_df['Price_Change'])

In [12]:
feature_df['Losses'] = np.where(feature_df['Price_Change'] > 0, 0, np.abs(feature_df['Price_Change']))

In [13]:
feature_df['Avg_Gain'] = feature_df.groupby('Ticker_Name')['Gains'].transform(lambda x: x.ewm(alpha=1/14, adjust=False).mean())

In [14]:
feature_df['Avg_Loss'] = feature_df.groupby('Ticker_Name')['Losses'].transform(lambda x: x.ewm(alpha= 1/14, adjust= False).mean())

In [15]:
feature_df.head(15)

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,Gains,Losses,Avg_Gain,Avg_Loss
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,NaN,NaN,NaN,NaN
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,0.000000,1.750000,0.000000,1.750000
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,97.749512,0.000000,6.982108,1.625000
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,0.000000,65.849609,6.483386,6.212472
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,29.450195,0.000000,8.123872,5.768724
5,2016-06-20,8238.500000,8244.150391,8107.350098,8115.750000,168600,^NSEI,8219.900391,0,68.299805,68.299805,0.000000,12.422153,5.356672
6,2016-06-21,8219.900391,8257.250000,8202.150391,8255.400391,137000,^NSEI,8203.700195,0,-18.599609,0.000000,18.599609,11.534857,6.302596
7,2016-06-22,8203.700195,8238.349609,8153.250000,8213.650391,136100,^NSEI,8270.450195,1,-16.200195,0.000000,16.200195,10.710938,7.009568
8,2016-06-23,8270.450195,8285.599609,8188.299805,8201.150391,154100,^NSEI,8088.600098,0,66.750000,66.750000,0.000000,14.713728,6.508884
9,2016-06-24,8088.600098,8100.700195,7927.049805,8029.100098,297600,^NSEI,8094.700195,1,-181.850098,0.000000,181.850098,13.662748,19.033257


In [16]:
feature_df['RS'] = feature_df['Avg_Gain'] / feature_df['Avg_Loss']

In [17]:
feature_df['RSI'] = 100 - (100 / (1+ feature_df['RS']))

In [18]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,Gains,Losses,Avg_Gain,Avg_Loss,RS,RSI
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,0.000000,1.750000,0.000000,1.750000,0.000000,0.000000
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,97.749512,0.000000,6.982108,1.625000,4.296682,81.120255
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,0.000000,65.849609,6.483386,6.212472,1.043608,51.066938
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,29.450195,0.000000,8.123872,5.768724,1.408262,58.476271


In [19]:
feature_df['Body_of_Candle'] = np.abs(feature_df['Open'] - feature_df['Close'])

In [20]:
feature_df['Range_of_Candle'] = feature_df['High'] - feature_df['Low']

In [21]:
feature_df['Doji'] = np.where(feature_df['Body_of_Candle'] <= ((feature_df['Range_of_Candle']) * 0.1), 1, 0)

In [22]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,Gains,Losses,Avg_Gain,Avg_Loss,RS,RSI,Body_of_Candle,Range_of_Candle,Doji
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.350098,61.350098,0
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,0.000000,1.750000,0.000000,1.750000,0.000000,0.000000,25.549805,65.450195,0
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,97.749512,0.000000,6.982108,1.625000,4.296682,81.120255,67.199707,90.050293,0
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,0.000000,65.849609,6.483386,6.212472,1.043608,51.066938,39.899902,106.199707,0
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,29.450195,0.000000,8.123872,5.768724,1.408262,58.476271,6.449707,59.450195,0


In [23]:
feature_df['Prev_Open'] = feature_df.groupby('Ticker_Name')['Open'].shift(1)

In [24]:
feature_df['Prev_Close'] = feature_df.groupby('Ticker_Name')['Close'].shift(1)

In [25]:
feature_df['Bullish_Engulfing'] = np.where(((feature_df['Prev_Close'] < feature_df['Prev_Open'])& 
                                            (feature_df['Close'] > feature_df['Open'])&
                                            (feature_df['Close'] > feature_df['Prev_Open'])&
                                            (feature_df['Open'] < feature_df['Prev_Close'])), 1, 0)

In [26]:
feature_df['Bearish_Engulfing'] = np.where(((feature_df['Prev_Close'] > feature_df['Prev_Open'])&
                                            (feature_df['Close'] < feature_df['Open'])&
                                            (feature_df['Open'] > feature_df['Prev_Close'])&
                                            (feature_df['Close'] < feature_df['Prev_Open'])), 1, 0)

In [27]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,...,Avg_Loss,RS,RSI,Body_of_Candle,Range_of_Candle,Doji,Prev_Open,Prev_Close,Bullish_Engulfing,Bearish_Engulfing
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,...,NaN,NaN,NaN,8.350098,61.350098,0,NaN,NaN,0,0
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,...,1.750000,0.000000,0.000000,25.549805,65.450195,0,8102.250000,8110.600098,0,0
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,...,1.625000,4.296682,81.120255,67.199707,90.050293,0,8134.399902,8108.850098,0,0
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,...,6.212472,1.043608,51.066938,39.899902,106.199707,0,8139.399902,8206.599609,0,0
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,...,5.768724,1.408262,58.476271,6.449707,59.450195,0,8180.649902,8140.750000,0,0


In [28]:
feature_df['Lower_Shadow'] = (np.minimum(feature_df['Open'], feature_df['Close'])) - feature_df['Low']

In [29]:
feature_df['Upper_Shadow'] = feature_df['High'] - (np.maximum(feature_df['Open'], feature_df['Close']))

In [30]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,...,RSI,Body_of_Candle,Range_of_Candle,Doji,Prev_Open,Prev_Close,Bullish_Engulfing,Bearish_Engulfing,Lower_Shadow,Upper_Shadow
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,...,NaN,8.350098,61.350098,0,NaN,NaN,0,0,38.350098,14.649902
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,...,0.000000,25.549805,65.450195,0,8102.250000,8110.600098,0,0,39.350098,0.550293
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,...,81.120255,67.199707,90.050293,0,8134.399902,8108.850098,0,0,16.250000,6.600586
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,...,51.066938,39.899902,106.199707,0,8139.399902,8206.599609,0,0,66.299805,0.000000
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,...,58.476271,6.449707,59.450195,0,8180.649902,8140.750000,0,0,34.400391,18.600098


In [31]:
feature_df['Hammer'] = np.where((feature_df['Lower_Shadow'] >= (2* feature_df['Body_of_Candle']))& 
                                (feature_df['Upper_Shadow'] <= feature_df['Body_of_Candle']), 1, 0)

In [32]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,...,Body_of_Candle,Range_of_Candle,Doji,Prev_Open,Prev_Close,Bullish_Engulfing,Bearish_Engulfing,Lower_Shadow,Upper_Shadow,Hammer
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,8108.850098,0,NaN,...,8.350098,61.350098,0,NaN,NaN,0,0,38.350098,14.649902,0
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,8206.599609,1,-1.750000,...,25.549805,65.450195,0,8102.250000,8110.600098,0,0,39.350098,0.550293,0
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,8140.750000,0,97.749512,...,67.199707,90.050293,0,8134.399902,8108.850098,0,0,16.250000,6.600586,0
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,8170.200195,1,-65.849609,...,39.899902,106.199707,0,8139.399902,8206.599609,0,0,66.299805,0.000000,0
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,8238.500000,1,29.450195,...,6.449707,59.450195,0,8180.649902,8140.750000,0,0,34.400391,18.600098,0


In [33]:
feature_df['EMA_12'] = feature_df['Close'].ewm(span=12, adjust=False).mean()

In [34]:
feature_df['EMA_26'] = feature_df['Close'].ewm(span=26, adjust=False).mean()

In [35]:
feature_df['MACD'] = feature_df['EMA_12'] - feature_df['EMA_26']

In [36]:
feature_df['MACD_Signal'] = feature_df['MACD'].ewm(span=9, adjust=False).mean()

In [37]:
feature_df.tail()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Next_Close,Target,Price_Change,...,Prev_Close,Bullish_Engulfing,Bearish_Engulfing,Lower_Shadow,Upper_Shadow,Hammer,EMA_12,EMA_26,MACD,MACD_Signal
19747,2026-06-05,3953.199951,3978.000000,3923.899902,3960.000000,1278243,LT.NS,3875.500000,0,11.099854,...,3942.100098,0,0,29.300049,18.000000,0,3976.057009,3959.086786,16.970222,22.105609
19748,2026-06-08,3875.500000,3916.699951,3863.100098,3901.100098,1678709,LT.NS,3900.600098,1,-77.699951,...,3953.199951,0,0,12.399902,15.599854,0,3960.586700,3952.895172,7.691527,19.222792
19749,2026-06-09,3900.600098,3924.899902,3885.000000,3924.899902,1506501,LT.NS,3917.500000,1,25.100098,...,3875.500000,0,0,15.600098,0.000000,0,3951.357992,3949.021463,2.336528,15.845540
19750,2026-06-10,3917.500000,3962.800049,3900.000000,3904.000000,2033070,LT.NS,3883.800049,0,16.899902,...,3900.600098,0,0,4.000000,45.300049,0,3946.149070,3946.686540,-0.537470,12.568938
19751,2026-06-11,3883.800049,3909.800049,3882.000000,3900.000000,222894,LT.NS,NaN,0,-33.699951,...,3917.500000,0,0,1.800049,9.800049,0,3936.556913,3942.028281,-5.471369,8.960876


In [38]:
feature_df.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker_Name',
       'Next_Close', 'Target', 'Price_Change', 'Gains', 'Losses', 'Avg_Gain',
       'Avg_Loss', 'RS', 'RSI', 'Body_of_Candle', 'Range_of_Candle', 'Doji',
       'Prev_Open', 'Prev_Close', 'Bullish_Engulfing', 'Bearish_Engulfing',
       'Lower_Shadow', 'Upper_Shadow', 'Hammer', 'EMA_12', 'EMA_26', 'MACD',
       'MACD_Signal'],
      dtype='str')

In [39]:
columns_to_drop = ['Next_Close', 'Price_Change', 'Gains', 'Losses', 'Avg_Gain', 'Avg_Loss', 'RS',
                   'Body_of_Candle', 'Range_of_Candle', 'Prev_Open', 'Prev_Close','Lower_Shadow', 'Upper_Shadow']

In [40]:
feature_df.drop(columns_to_drop, axis= 1, inplace= True)

In [41]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Target,RSI,Doji,Bullish_Engulfing,Bearish_Engulfing,Hammer,EMA_12,EMA_26,MACD,MACD_Signal
0,2016-06-13,8110.600098,8125.250000,8063.899902,8102.250000,169700,^NSEI,0,NaN,0,0,0,0,8110.600098,8110.600098,0.000000,0.000000
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,1,0.000000,0,0,0,0,8110.330867,8110.470468,-0.139601,-0.027920
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,0,81.120255,0,0,0,0,8125.141443,8117.591145,7.550297,1.487723
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,1,51.066938,0,0,0,0,8127.542759,8119.306616,8.236143,2.837407
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,1,58.476271,0,0,0,0,8134.105442,8123.076511,11.028931,4.475712


In [42]:
feature_df.dropna(inplace= True)

In [43]:
feature_df.head()

,Date,Close,High,Low,Open,Volume,Ticker_Name,Target,RSI,Doji,Bullish_Engulfing,Bearish_Engulfing,Hammer,EMA_12,EMA_26,MACD,MACD_Signal
1,2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,^NSEI,1,0.000000,0,0,0,0,8110.330867,8110.470468,-0.139601,-0.027920
2,2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,^NSEI,0,81.120255,0,0,0,0,8125.141443,8117.591145,7.550297,1.487723
3,2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,^NSEI,1,51.066938,0,0,0,0,8127.542759,8119.306616,8.236143,2.837407
4,2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,^NSEI,1,58.476271,0,0,0,0,8134.105442,8123.076511,11.028931,4.475712
5,2016-06-20,8238.500000,8244.150391,8107.350098,8115.750000,168600,^NSEI,0,69.870494,0,1,0,0,8150.166143,8131.626399,18.539744,7.288518


In [44]:
feature_df = pd.get_dummies(data= feature_df, columns= ['Ticker_Name'], drop_first= True ,dtype= int)

In [45]:
feature_df.set_index('Date', inplace= True)

In [46]:
feature_df.head()

,Close,High,Low,Open,Volume,Target,RSI,Doji,Bullish_Engulfing,Bearish_Engulfing,...,EMA_26,MACD,MACD_Signal,Ticker_Name_INFY.NS,Ticker_Name_ITC.NS,Ticker_Name_LT.NS,Ticker_Name_RELIANCE.NS,Ticker_Name_TCS.NS,Ticker_Name_^NSEBANK,Ticker_Name_^NSEI
Date,,,,,,,,,,,,,,,,,,,,,
2016-06-14,8108.850098,8134.950195,8069.500000,8134.399902,145500,1,0.000000,0,0,0,...,8110.470468,-0.139601,-0.027920,0,0,0,0,0,0,1
2016-06-15,8206.599609,8213.200195,8123.149902,8139.399902,169800,0,81.120255,0,0,0,...,8117.591145,7.550297,1.487723,0,0,0,0,0,0,1
2016-06-16,8140.750000,8180.649902,8074.450195,8180.649902,189200,1,51.066938,0,0,0,...,8119.306616,8.236143,2.837407,0,0,0,0,0,0,1
2016-06-17,8170.200195,8195.250000,8135.799805,8176.649902,166600,1,58.476271,0,0,0,...,8123.076511,11.028931,4.475712,0,0,0,0,0,0,1
2016-06-20,8238.500000,8244.150391,8107.350098,8115.750000,168600,0,69.870494,0,1,0,...,8131.626399,18.539744,7.288518,0,0,0,0,0,0,1


In [47]:
feature_df.to_csv('../data/processed/engineered_market_data.csv', index=True)